In [2]:
import numpy as np
import math
import re
import itertools
import pandas as pd
import time
import os
import pickle
import copy
from Functions import *
from Fast_functions import *
import cmcrameri.cm as cmc

In [5]:
diff=3
max_checks=1000
n_compounds=100
scrambler={1:np.arange(n_compounds)}
for j in range(2,diff+1):
    scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})

In [6]:
def decode_precomp(well_assigner:np.array, differentiate:int, 
                   scrambler:dict, readout:np.ndarray, max_differentiate=-1, **kwargs) -> list:
    if differentiate==0:
        return(True,well_assigner, np.array([1]*well_assigner.shape[0]))
    N=well_assigner.shape[0]
    sc_list=np.arange(N).tolist()
    for i in range(differentiate):
        diff=i+1
        if diff ==1:
            full_well_assigner=well_assigner.copy()
        else:
            this_sc=scrambler[diff]
            #print(this_sc)
            #print(well_assigner)
            #print(diff)
            full_well_assigner=np.concatenate((full_well_assigner,np.any(well_assigner[this_sc], axis=1)))
            sc_list.extend(this_sc.tolist())
    #outcomes,_=np.unique(full_well_assigner, axis=0, return_counts=True)
    idxs = np.all(readout == full_well_assigner, axis=1)
    return list(itertools.compress(sc_list,idxs))
        

In [15]:

def fast_decode(well_assigner:np.array, differentiate:int, readout:np.ndarray, 
                max_checks=1e4, **kwargs):
    WA=well_assigner
    n_pools=WA.shape[1]


    if np.max(readout)>1 or len(readout)!=n_pools:
        readout_bin_ls = [1 if i in readout else 0 for i in range(n_compounds)]
        readout_bl=np.array(readout_bin_ls)
    else:
        readout_bl=readout
    mask = ~np.any((WA == 1) & (readout_bl == 0), axis=1)
                    #msg+=f'boolean readout {readout_bl}<br>'
    original_indices = np.where(mask)[0]  # Get original row indices
    filtered_WA = WA[mask]
    n_compounds=filtered_WA.shape[0]
    if n_compounds<differentiate:
        differentiate=n_compounds
    if n_compounds<2:
        if n_compounds==1:
            decoded=[original_indices[0]] 
        else:
            decoded=[]

    else:
        MC=0
        MP=1
        ls_combs=[]
        ls_diffs=[]
        difo=differentiate
        while (MC<max_checks or MP>mp) and difo>0:
            MCI=math.comb(n_compounds,differentiate)
            MC+=MCI
            mp=MCI/MC
            ls_combs.append(MCI)
            ls_diffs.append(difo)
            difo-=1
        if MC>max_checks:
            decoded = [int(original_indices[idx]) for idx in range(len(original_indices))]
        
        else:
            scrambler={1:np.arange(n_compounds)}
            for j in range(2,differentiate+1):
                scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})
            decoded_pre=decode_precomp(well_assigner=filtered_WA, differentiate= differentiate, scrambler=scrambler, 
                    readout=readout_bl)
            # Map filtered indices back to original indices
            decoders = [combination if isinstance(combination, list) else [combination] for combination in decoded_pre]
            decoded = [[int(original_indices[idx]) for idx in combination] for combination in decoders]
        
    return decoded




In [ ]:

def fast_decode(well_assigner:np.array, differentiate:int, readout:np.ndarray, 
                max_checks=1e4, **kwargs):
    WA=well_assigner
    n_pools=WA.shape[1]


    if np.max(readout)>1 or len(readout)!=n_pools:
        readout_bin_ls = [1 if i in readout else 0 for i in range(n_compounds)]
        readout_bl=np.array(readout_bin_ls)
    else:
        readout_bl=readout
    mask = ~np.any((WA == 1) & (readout_bl == 0), axis=1)
                    #msg+=f'boolean readout {readout_bl}<br>'
    original_indices = np.where(mask)[0]  # Get original row indices
    filtered_WA = WA[mask]
    n_compounds=filtered_WA.shape[0]
    if n_compounds<differentiate:
        differentiate=n_compounds
    if n_compounds<2:
        if n_compounds==1:
            decoded=[original_indices[0]] 
        else:
            decoded=[]

    else:
        MC=0
        MP=1
        ls_combs=[]
        ls_diffs=[]
        difo=differentiate
        while (MC<max_checks or MP>mp) and difo>0:
            MCI=math.comb(n_compounds,differentiate)
            MC+=MCI
            mp=MCI/MC
            ls_combs.append(MCI)
            ls_diffs.append(difo)
            difo-=1
        if MC>max_checks:
            decoded = [int(original_indices[idx]) for idx in range(len(original_indices))]
        
        else:
            scrambler={1:np.arange(n_compounds)}
            for j in range(2,differentiate+1):
                scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})
            decoded_pre=decode_precomp(well_assigner=filtered_WA, differentiate= differentiate, scrambler=scrambler, 
                    readout=readout_bl)
            # Map filtered indices back to original indices
            decoders = [combination if isinstance(combination, list) else [combination] for combination in decoded_pre]
            decoded = [[int(original_indices[idx]) for idx in combination] for combination in decoders]
        
    return decoded


def is_consistent_precomp(well_assigner:np.array, differentiate:int, scrambler:dict) -> list:
    if differentiate==0:
        return(True,well_assigner, np.array([1]*well_assigner.shape[0]))
    N=well_assigner.shape[0]
    for i in range(differentiate):
        diff=i+1
        if diff ==1:
            full_well_assigner=well_assigner.copy()
        else:
            this_sc=scrambler[diff]
            #print(this_sc)
            #print(well_assigner)
            #print(diff)
            
            full_well_assigner=np.concatenate((full_well_assigner,np.any(well_assigner[this_sc], axis=1)))
    _, counts=np.unique(full_well_assigner, axis=0, return_counts=True)
    if len(counts)<full_well_assigner.shape[0]:
        return(False, full_well_assigner, counts)
    elif len(counts)==full_well_assigner.shape[0]:
        return(True,full_well_assigner, counts)
    else:
        print("Something is fishy")
        return(-1)
    
def mean_metrics_precomp(well_assigner, differentiate, scrambler, **kwargs):
    BT=well_assigner.shape[1]
    _,_, counts= is_consistent_precomp(well_assigner, differentiate, scrambler) 
    ET=extra_tests(counts)  
    ET =ET if ET<well_assigner.shape[0] else well_assigner.shape[0]
    ER=np.sum(counts[counts>1])/np.sum(counts)
    rounds=ER+1
    p_check=np.round(ER*100)
    return BT+ET, ET,  rounds, p_check

def mean_metrics_fast(well_assigner, differentiate, max_checks=1e4, scaler=1, mp=1e-5, **kwargs):
    BT=well_assigner.shape[1]
    n_compounds=well_assigner.shape[0]
    MC=0
    MP=1
    ls_combs=[]
    ls_diffs=[]
    difo=differentiate
    while (MC<max_checks or MP>mp) and difo>0:
        MCI=math.comb(n_compounds,differentiate)
        MC+=MCI
        mp=MCI/MC
        ls_combs.append(MCI)
        ls_diffs.append(difo)
        difo-=1
        

    if MC>max_checks:
        counts=[]
        probi=np.array(ls_combs, dtype=float)
        probi/=np.sum(probi)
        differis=np.random.choice(ls_diffs, int(max_checks*scaler), p=probi)
        #differo=differentiate
        #for _ in range(int(max_checks*scaler)):
        for differo in differis:
            rnd_pos=np.random.choice(np.arange(n_compounds), differo, replace=False)
            readout=np.any(well_assigner[rnd_pos], axis=0)
            decoded=fast_decode(well_assigner=well_assigner, differentiate=differo, 
                                readout=readout, max_checks=int(max_checks/10+5))
            counts.append(len(decoded))
        counts=np.array(counts)
        ET=np.sum(counts-1)/len(counts)
        ET =ET if ET<well_assigner.shape[0] else well_assigner.shape[0]
        ER=np.sum(counts>1)/np.sum(counts>0)
        rounds=ER+1
        p_check=np.round(ER*100)
        return BT+ET, ET,  rounds, p_check
    else:
        scrambler={1:np.arange(n_compounds)}
        for j in range(2,differentiate+1):
            scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})
        
        return mean_metrics_precomp(well_assigner, differentiate, scrambler, **kwargs)
                


In [ ]:


def rand_sweep_diff_fast(n_compounds, max_diff, Npath, **kwargs):
    if 'differentiate' in kwargs.keys():
        del kwargs['differentiate']
    N=n_compounds
    
    if max_diff>1:
        
        dpath=os.path.join(Npath,'diff_'+str(max_diff))
        WApath=os.path.join(dpath,'WAs')
        if not kwargs['overwrite'] and check_Rand_in_WApath(WApath):
            return
        
        
        for di in range(max_diff):
            start_time = time.time()
            diff=di+1
            if diff==1:
                dpath=os.path.join(Npath,'diff_'+str(diff))
                WApath=os.path.join(dpath,'WAs')
                scrambler={1:np.arange(n_compounds)}

                if not kwargs['overwrite'] and check_Rand_in_WApath(WApath):
                    continue

                
                WA_rand,  min_tests, perc_check=assign_wells_random_precomp(n_compounds=n_compounds, 
                                                                differentiate=diff,scrambler=scrambler, return_me=True, **kwargs )
                extra_exp=WA_rand.shape[1]+min_tests

                if kwargs['cleanup']=='one_liner' or kwargs['cleanup']=='full' or kwargs['cleanup']=='True':
                    filenames = next(os.walk(WApath), (None, None, []))[2]
                    for fname in filenames:
                        if fname.startswith('WA_Random_N_'):
                            os.remove(os.path.join(WApath,fname))

                if kwargs['cleanup']=='WA' or kwargs['cleanup']=='full' or kwargs['cleanup']=='True':
                    filenames = next(os.walk(dpath), (None, None, []))[2]
                    for fname in filenames:
                        if fname.startswith('Random_diff_'):
                            os.remove(os.path.join(dpath,fname))

                #.append(['Random', min_tests, np.max(np.sum(WA_rand, axis=0)), WA_rand.shape[0], int(perc_check),  extra_exp,1+perc_check/100])
                full_file_dir=os.path.join(dpath,'Random_diff_'+str(diff)+'_NS_'+
                                               str(n_compounds)+'_NW_'+str(WA_rand.shape[1])+
                                               '_MS_'+str(np.max(np.sum(WA_rand, axis=0)))+
                                                '_PC_'+ str(int(perc_check)) +'_ME_'+str(np.round(min_tests,2))+".txt")
                if not os.path.exists(dpath):
                    os.makedirs(dpath)
                if kwargs['one_liner']:
                    open(full_file_dir, 'a').close()

                if not os.path.exists(WApath):
                    os.makedirs(WApath)
                thisfile=os.path.join(WApath,'WA_Random_N_'+str(n_compounds)+'_diff_'+str(diff)+
                                      '_ME_'+str(np.round(min_tests,2))+'.csv')
                np.savetxt(thisfile, WA_rand.astype(bool), delimiter=",")


            else:
                dpath=os.path.join(Npath,'diff_'+str(diff))
                WApath=os.path.join(dpath,'WAs')
                this_sc_file=os.path.join(dir_scramblers, 'N_'+str(N),  'N_'+str(N)+'_diff_'+str(diff)+'.npz')
                this_scrambler=np.load(this_sc_file)['sc']
                scrambler.update({diff:this_scrambler})

                if not kwargs['overwrite'] and check_Rand_in_WApath(WApath):
                    continue


                WA_rand,  min_tests, perc_check=assign_wells_random_precomp(n_compounds=n_compounds, 
                                                                differentiate=diff,scrambler=scrambler, return_me=True, **kwargs  )
                extra_exp=WA_rand.shape[1]+min_tests

                if kwargs['cleanup']=='one_liner' or kwargs['cleanup']=='full' or kwargs['cleanup']=='True':
                    filenames = next(os.walk(WApath), (None, None, []))[2]
                    for fname in filenames:
                        if fname.startswith('WA_Random_N_'):
                            os.remove(os.path.join(WApath,fname))

                if kwargs['cleanup']=='WA' or kwargs['cleanup']=='full' or kwargs['cleanup']=='True':
                    filenames = next(os.walk(dpath), (None, None, []))[2]
                    for fname in filenames:
                        if fname.startswith('Random_diff_'):
                            os.remove(os.path.join(dpath,fname))


                #.append(['Random', min_tests, np.max(np.sum(WA_rand, axis=0)), WA_rand.shape[0], int(perc_check),  extra_exp,1+perc_check/100])
                full_file_dir=os.path.join(dpath,'Random_diff_'+str(diff)+'_NS_'+
                                               str(n_compounds)+'_NW_'+str(WA_rand.shape[1])+
                                               '_MS_'+str(np.max(np.sum(WA_rand, axis=0)))+
                                                '_PC_'+ str(int(perc_check)) +'_ME_'+str(np.round(min_tests,2))+".txt")
                


                if not os.path.exists(dpath):
                    os.makedirs(dpath)

                if kwargs['one_liner']:
                    open(full_file_dir, 'a').close()
                if not os.path.exists(WApath):
                    os.makedirs(WApath)
                thisfile=os.path.join(WApath,'WA_Random_N_'+str(n_compounds)+'_diff_'+str(diff)+
                                      '_ME_'+str(np.round(min_tests,2))+'.csv')
                np.savetxt(thisfile, WA_rand.astype(bool), delimiter=",")

            DTS=np.round((time.time() - start_time),2)
            DTD=DTS//86400
            DTH=DTS//3600-DTD*24
            DTM=DTS//60-DTH*60-DTD*24*60
            DTS=np.round(DTS-(DTM+DTH*60+DTD*24*60)*60,2)
            print("%s days %s hours %s minutes and %s seconds required for N= %s and differentiate %s" % 
                  (DTD, DTH, DTM, DTS, n_compounds, diff))
            print('----------------------------------------------------------------------------------------------------------') 

    elif max_diff==1:

        start_time = time.time()
        diff=1
        dpath=os.path.join(Npath,'diff_'+str(diff))
        WApath=os.path.join(dpath,'WAs')

        if not kwargs['overwrite'] and check_Rand_in_WApath(WApath):
            return

        scrambler={1:np.arange(n_compounds)}
        WA_rand,  min_tests, perc_check=assign_wells_random_precomp(n_compounds=n_compounds, 
                                                        differentiate=diff,scrambler=scrambler, return_me=True, **kwargs )
        extra_exp=WA_rand.shape[1]+min_tests
        #.append(['Random', min_tests, np.max(np.sum(WA_rand, axis=0)), WA_rand.shape[0], int(perc_check),  extra_exp,1+perc_check/100])
        
        
        if kwargs['cleanup']=='one_liner' or kwargs['cleanup']=='full' or kwargs['cleanup']=='True':
            filenames = next(os.walk(WApath), (None, None, []))[2]
            for fname in filenames:
                if fname.startswith('WA_Random_N_'):
                    os.remove(os.path.join(WApath,fname))

        if kwargs['cleanup']=='WA' or kwargs['cleanup']=='full' or kwargs['cleanup']=='True':
            filenames = next(os.walk(dpath), (None, None, []))[2]
            for fname in filenames:
                if fname.startswith('Random_diff_'):
                    os.remove(os.path.join(dpath,fname))
        
        
        
        full_file_dir=os.path.join(dpath,'Random_diff_'+str(diff)+'_NS_'+
                                        str(n_compounds)+'_NW_'+str(WA_rand.shape[1])+
                                        '_MS_'+str(np.max(np.sum(WA_rand, axis=0)))+
                                        '_PC_'+ str(int(perc_check)) +'_ME_'+str(np.round(min_tests,2))+".txt")


        if not os.path.exists(dpath):
            os.makedirs(dpath)
        if kwargs['one_liner']:
            open(full_file_dir, 'a').close()
        if not os.path.exists(WApath):
            os.makedirs(WApath)
        thisfile=os.path.join(WApath,'WA_Random_N_'+str(n_compounds)+'_diff_'+str(diff)+
                                '_ME_'+str(np.round(min_tests,2))+'.csv')
        np.savetxt(thisfile, WA_rand.astype(bool), delimiter=",")


        DTS=np.round((time.time() - start_time),2)
        DTD=DTS//86400
        DTH=DTS//3600-DTD*24
        DTM=DTS//60-DTH*60-DTD*24*60
        DTS=np.round(DTS-(DTM+DTH*60+DTD*24*60)*60,2)
        print("%s days %s hours %s minutes and %s seconds required for N= %s and differentiate %s" % 
                (DTD, DTH, DTM, DTS, n_compounds, diff))
        print('----------------------------------------------------------------------------------------------------------') 



In [17]:
differo=15
compi=5000

In [18]:
WA_mat=assign_wells_mat(n_compounds=compi)

In [19]:
rnd_pos=np.random.choice(np.arange(compi), differo, replace=True)

In [20]:
readout=np.any(WA_mat[rnd_pos], axis=0)

In [35]:
for _ in range(1000):
    mean_metrics_fast(WA_mat, differo, max_checks=1000, scaler=0.3)

In [23]:
mean_metrics_fast(WA_mat, differo)

(326.19100000000003, 184.191, 2.0, 100.0)

In [ ]:
scramblero={1:np.arange(compi)}
for j in range(2,differo+1):
    scramblero.update({j:np.array(list(itertools.combinations(np.arange(compi),j)))})
mean_metrics_precomp(WA_mat, differo, scramblero)


In [ ]:
2/9


In [ ]:
[math.comb(compi,i+1) for i in range(differo)]

In [ ]:
rout=np.sum(WA_mat[np.array([1])], axis=0).astype(bool)

In [ ]:
fast_decode(well_assigner=WA_mat, differentiate=1, 
                            readout=rout)

In [ ]:
n_compounds=85
diff=2

In [ ]:
obji=[1,2,3,4,5]
[list(x) for x in itertools.combinations(obji, 2)]

In [ ]:
scrambler={1:np.arange(n_compounds)}
for j in range(2,diff+1):
    scrambler.update({j:np.array(list(itertools.combinations(np.arange(n_compounds),j)))})

In [ ]:
WA_mat[scrambler[2]].shape

In [ ]:
math.comb(n_compounds,50)

In [ ]:
stirl_binom_extra_approx(n_compounds,50)

array([0, 1, 2, 3, 4, 5, 6])